In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
import ast
import json

In [5]:
df = pd.read_csv('merged.csv')
df.head(5)

,subject_id,hadm_id,icustay_id,dbsource,first_careunit,last_careunit,first_wardid,last_wardid,intime,outtime,...,expire_flag,icd9_codes,diagnosis_count,drugs,prescription_count,services,lab_count,abnormal_lab_count,transfer_count,procedure_count
0,10006,142345,206504,carevue,MICU,MICU,52,52,2164-10-23 21:10:15,2164-10-25 12:21:07,...,1,"['99591', '99662', '5672', '40391', '42731', '...",21,"['Magnesium Oxide', 'Docusate Sodium', 'Senna'...",18.0,['MED'],251,93,3,7.0
1,10011,105331,232110,carevue,MICU,MICU,15,15,2126-08-14 22:34:00,2126-08-28 18:59:00,...,1,"['570', '07030', '07054', '30401', '2875', '27...",6,NaN,NaN,['MED'],700,252,2,2.0
2,10013,165520,264446,carevue,MICU,MICU,15,15,2125-10-04 23:38:00,2125-10-07 15:13:52,...,1,"['0389', '41071', '78551', '486', '42731', '20...",9,"['Furosemide', 'Azithromycin', 'Azithromycin',...",36.0,['MED'],148,38,2,1.0
3,10017,199207,204881,carevue,CCU,CCU,7,7,2149-05-29 18:52:29,2149-05-31 22:19:17,...,1,"['81201', '4928', '8028', '8024', '99812', '41...",14,"['Potassium Chloride', 'Neutra-Phos', 'Oxycodo...",20.0,['MED'],302,109,6,2.0
4,10019,177759,228977,carevue,MICU,MICU,15,15,2163-05-14 20:43:56,2163-05-16 03:47:04,...,1,"['0389', '51881', '5770', '30390', '5781', '58...",14,"['Albumin 25% (12.5gm)', 'Dexamethasone', 'Cal...",67.0,['MED'],287,142,2,4.0


In [25]:
df['hospital_expire_flag']

0      0
1      1
2      1
3      0
4      1
      ..
131    0
132    1
133    0
134    0
135    0
Name: hospital_expire_flag, Length: 136, dtype: int64

In [7]:
# Get age
df['dob'] = pd.to_datetime(df['dob'])
df['admittime'] = pd.to_datetime(df['admittime'])
df['age'] = df['admittime'].dt.year - df['dob'].dt.year
df['age'] = df['age'].clip(upper=90)

In [ ]:
# Length of stay fill
df['los'] = df['los'].fillna(df['los'].median())

In [13]:
df['abnormal_lab_ratio'] = np.where(
    df['lab_count'] > 0,
    df['abnormal_lab_count'] / df['lab_count'],
    0
)

In [14]:
df['prescription_count'] = df['prescription_count'].fillna(0)
 

In [15]:
def count_list_items(val):
    if pd.isna(val) or val == '':
        return 0
    try:
        return len(ast.literal_eval(val))
    except:
        return 0

In [16]:
df['unique_drug_count'] = df['drugs'].apply(count_list_items)
df['unique_icd9_count'] = df['icd9_codes'].apply(count_list_items)

In [20]:
df['is_male'] = (df['gender'] == 'M').astype(int)

In [17]:
df['service_count'] = df['services'].apply(count_list_items)

In [18]:
df['is_emergency'] = (df['admission_type'] == 'EMERGENCY').astype(int)

In [22]:
def simplify_ethnicity(eth):
    if pd.isna(eth): return 'UNKNOWN'
    eth = eth.upper()
    if 'WHITE' in eth: return 'WHITE'
    if 'BLACK' in eth: return 'BLACK'
    if 'ASIAN' in eth: return 'ASIAN'
    if 'HISPANIC' in eth: return 'HISPANIC'
    return 'OTHER'
 
df['ethnicity_simple'] = df['ethnicity'].apply(simplify_ethnicity)
eth_dummies = pd.get_dummies(df['ethnicity_simple'], prefix='eth', drop_first=True)
df = pd.concat([df, eth_dummies], axis=1)
 
# Sepsis flag from diagnosis text
df['has_sepsis'] = df['diagnosis'].str.upper().str.contains('SEPSIS', na=False).astype(int)
 
# Transferred in
df['is_transfer'] = df['admission_location'].str.contains('TRANSFER', na=False).astype(int)

In [24]:
feature_cols = [
    'age', 'is_male', 'los', 'diagnosis_count', 'unique_icd9_count',
    'prescription_count', 'unique_drug_count', 'lab_count', 'abnormal_lab_count',
    'abnormal_lab_ratio', 'transfer_count', 'procedure_count', 'service_count',
    'is_emergency','has_sepsis', 'is_transfer'
] + [c for c in df.columns if c.startswith('eth_')]
 
X = df[feature_cols].copy()
y = df['hospital_expire_flag'].values
 
print(f"Feature matrix: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Features: {list(X.columns)}\n")

Feature matrix: 136 samples × 20 features
Features: ['age', 'is_male', 'los', 'diagnosis_count', 'unique_icd9_count', 'prescription_count', 'unique_drug_count', 'lab_count', 'abnormal_lab_count', 'abnormal_lab_ratio', 'transfer_count', 'procedure_count', 'service_count', 'is_emergency', 'has_sepsis', 'is_transfer', 'eth_BLACK', 'eth_HISPANIC', 'eth_OTHER', 'eth_WHITE']



In [30]:
# Select features and target variable
X = df[feature_cols].copy()
y = df['hospital_expire_flag'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
models = {
    'Logistic Regression': LogisticRegression(
        C=0.5, penalty='l2', max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=5, min_samples_leaf=5,
        random_state=42, class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=5, subsample=0.8, random_state=42
    ),
}
 
results = {}
all_probas = {}
 
for name, model in models.items():
    probas = cross_val_predict(model, X_scaled, y, cv=cv, method='predict_proba')[:, 1]
 
    auc_score = roc_auc_score(y, probas)
    brier = brier_score_loss(y, probas)
 
    frac_pos, mean_pred = calibration_curve(y, probas, n_bins=5, strategy='uniform')
 
    results[name] = {
        'auc_roc': round(auc_score, 4),
        'brier_score': round(brier, 4),
        'mean_score': round(probas.mean() * 100, 1),
        'std_score': round(probas.std() * 100, 1),
        'calibration': {
            'predicted': [round(x, 3) for x in mean_pred.tolist()],
            'observed': [round(x, 3) for x in frac_pos.tolist()]
        }
    }
    all_probas[name] = probas
 
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  AUC-ROC:  {auc_score:.4f}")
    print(f"  Brier Score:  {brier:.4f}")
    print(f"  Score range: {probas.min()*100:.1f} – {probas.max()*100:.1f}")
    print(f"  Score mean: {probas.mean()*100:.1f} ± {probas.std()*100:.1f}\n")
 

  Logistic Regression
  AUC-ROC (ranking quality):  0.6688
  Brier Score (calibration):  0.2327
  Score range: 1.6 – 99.1
  Score mean ± std: 45.7 ± 23.9

  Random Forest
  AUC-ROC (ranking quality):  0.7563
  Brier Score (calibration):  0.1908
  Score range: 10.2 – 81.9
  Score mean ± std: 41.3 ± 17.1

  Gradient Boosting
  AUC-ROC (ranking quality):  0.7524
  Brier Score (calibration):  0.2046
  Score range: 0.7 – 98.3
  Score mean ± std: 30.5 ± 30.7



In [ ]:
weights = {name: results[name]['auc_roc'] for name in models}
total_w = sum(weights.values())
weights = {k: v / total_w for k, v in weights.items()}
 
ensemble_probas = sum(all_probas[name] * weights[name] for name in models)
ensemble_auc = roc_auc_score(y, ensemble_probas)
ensemble_brier = brier_score_loss(y, ensemble_probas)
ensemble_cal_frac, ensemble_cal_mean = calibration_curve(y, ensemble_probas, n_bins=5, strategy='uniform')
 
print(f"  ENSEMBLE RISK SCORE")
print(f"  AUC-ROC:  {ensemble_auc:.4f}")
print(f"  Brier Score:  {ensemble_brier:.4f}")
print(f"  Weights: {json.dumps({k: round(v, 3) for k, v in weights.items()})}")
print(f"  Score range: {ensemble_probas.min()*100:.1f} – {ensemble_probas.max()*100:.1f}")
print(f"  Score mean ± std: {ensemble_probas.mean()*100:.1f} ± {ensemble_probas.std()*100:.1f}\n")


  ENSEMBLE RISK SCORE
  AUC-ROC (ranking quality):  0.7587
  Brier Score (calibration):  0.1874
  Weights: {"Logistic Regression": 0.307, "Random Forest": 0.347, "Gradient Boosting": 0.346}
  Score range: 7.0 – 83.5
  Score mean ± std: 38.9 ± 20.8



In [35]:
df['risk_score'] = (ensemble_probas * 100).round(1)
df['risk_category'] = pd.cut(
    df['risk_score'],
    bins=[0, 20, 40, 60, 80, 100],
    labels=['Low', 'Low-Moderate', 'Moderate', 'High', 'Critical'],
    include_lowest=True
)

In [36]:
df

,subject_id,hadm_id,icustay_id,dbsource,first_careunit,last_careunit,first_wardid,last_wardid,intime,outtime,...,is_male,ethnicity_simple,eth_BLACK,eth_HISPANIC,eth_OTHER,eth_WHITE,has_sepsis,is_transfer,risk_score,risk_category
0,10006,142345,206504,carevue,MICU,MICU,52,52,2164-10-23 21:10:15,2164-10-25 12:21:07,...,0,BLACK,True,False,False,False,1,0,23.3,Low-Moderate
1,10011,105331,232110,carevue,MICU,MICU,15,15,2126-08-14 22:34:00,2126-08-28 18:59:00,...,0,OTHER,False,False,True,False,0,1,83.5,Critical
2,10013,165520,264446,carevue,MICU,MICU,15,15,2125-10-04 23:38:00,2125-10-07 15:13:52,...,0,OTHER,False,False,True,False,1,1,45.3,Moderate
3,10017,199207,204881,carevue,CCU,CCU,7,7,2149-05-29 18:52:29,2149-05-31 22:19:17,...,0,WHITE,False,False,False,True,0,0,20.9,Low-Moderate
4,10019,177759,228977,carevue,MICU,MICU,15,15,2163-05-14 20:43:56,2163-05-16 03:47:04,...,1,WHITE,False,False,False,True,0,1,45.5,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,44083,198330,286428,metavision,CCU,CCU,7,7,2112-05-29 02:01:33,2112-06-01 16:50:40,...,1,WHITE,False,False,False,True,0,0,18.0,Low
132,44154,174245,217724,metavision,MICU,MICU,50,50,2178-05-14 20:29:55,2178-05-15 11:31:12,...,1,WHITE,False,False,False,True,0,0,52.4,Moderate
133,44212,163189,239396,metavision,MICU,MICU,50,50,2123-11-24 14:14:29,2123-12-25 17:12:19,...,0,BLACK,True,False,False,False,0,1,62.3,High
134,44222,192189,238186,metavision,CCU,CCU,7,7,2180-07-19 06:56:38,2180-07-20 14:48:45,...,1,WHITE,False,False,False,True,0,0,34.1,Low-Moderate
